In [ ]:
# Cell 1: Install Required Packages

!pip install transformers datasets torch evaluate nltk rouge-score -q


In [ ]:
# CELL 2: Import Libraries

import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    GPT2Config,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Check PyTorch and GPU availability
print("="*80)
print("ENVIRONMENT SETUP")
print("="*80)
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  Running on CPU (this will be slower)")
print("="*80)

In [ ]:
# CELL 3: Load WikiText-103 Dataset


print("\n" + "="*80)
print("LOADING DATASET")
print("="*80)

print("\nLoading WikiText-103 dataset...")
dataset = load_dataset("wikitext", "wikitext-103-v1")

# Display dataset structure
print("\n--- Dataset Structure ---")
print(dataset)

# Show sample from training set
print("\n--- Sample Text from Training Set ---")
sample_text = dataset['train'][100]['text']
print(sample_text[:500] if len(sample_text) > 500 else sample_text)

# Dataset statistics
print("\n--- Dataset Statistics ---")
print(f"Training samples: {len(dataset['train']):,}")
print(f"Validation samples: {len(dataset['validation']):,}")
print(f"Test samples: {len(dataset['test']):,}")
print("="*80)

In [ ]:
# CELL 4: Load GPT-2 Small Model and Tokenizer

print("\n" + "="*80)
print("LOADING GPT-2 MODEL")
print("="*80)

model_name = "gpt2"  # GPT-2 small (124M parameters)
print(f"\nLoading {model_name} tokenizer and model...")

# Load tokenizer (converts text to numbers)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have pad token by default

# Load pre-trained model
model = GPT2LMHeadModel.from_pretrained(model_name)

# Get model configuration
config = model.config

print("\n--- Model Configuration ---")
print(f"Model name: {model_name}")
print(f"Vocabulary size: {config.vocab_size:,}")
print(f"Hidden size: {config.n_embd}")
print(f"Number of layers: {config.n_layer}")
print(f"Number of attention heads: {config.n_head}")
print(f"Context window: {config.n_positions} tokens")

# Calculate total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n--- Model Size ---")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / (1024**2):.1f} MB (FP32)")
print("="*80)

In [ ]:
# CELL 5: Tokenize Dataset

print("\n" + "="*80)
print("DATA PREPROCESSING - TOKENIZATION")
print("="*80)

# Tokenization function
def tokenize_function(examples):
    """Convert text to token IDs"""
    return tokenizer(examples['text'], truncation=True, max_length=512)

# Tokenize all datasets
print("\nTokenizing datasets...")
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    desc="Tokenizing"
)

print("\n✓ Tokenization complete!")
print("\n--- Tokenized Dataset Structure ---")
print(tokenized_datasets)
print("="*80)

In [ ]:
# CELL 6: Group Texts into Fixed-Size Blocks

print("\n" + "="*80)
print("DATA PREPROCESSING - GROUPING INTO BLOCKS")
print("="*80)

block_size = 256  # Smaller blocks for baseline (faster processing)

def group_texts(examples):
    """
    Group tokens into fixed-size blocks of block_size tokens
    This is necessary for batch processing
    """
    # Concatenate all texts
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated['input_ids'])

    # Drop last chunk if it's smaller than block_size
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size

    # Split by chunks of block_size
    result = {
        k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    # Labels are the same as input_ids for language modeling
    result['labels'] = result['input_ids'].copy()
    return result

print(f"\nGrouping texts into {block_size}-token blocks...")

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    desc="Grouping texts"
)

print("\n Grouping complete!")
print("\n--- Preprocessed Dataset ---")
print(lm_datasets)
print(f"\nBlock size: {block_size} tokens")
print(f"Training blocks: {len(lm_datasets['train']):,}")
print(f"Validation blocks: {len(lm_datasets['validation']):,}")
print(f"Test blocks: {len(lm_datasets['test']):,}")
print("="*80)

In [ ]:
# CELL 7: Setup Data Collator and Move Model to GPU

print("\n" + "="*80)
print("EVALUATION SETUP")
print("="*80)

# Prepare data collator (handles batching)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM (not masked LM)
)

# Setup evaluation device (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

# Move model to device
model.to(device)
model.eval()  # Set to evaluation mode (disables dropout, etc.)

print("Model ready for evaluation")

In [ ]:
# CELL 8: Define Perplexity Calculation Function

def calculate_perplexity(model, dataset, max_samples=500):
    """
    Calculate perplexity on a subset of the dataset.

    Perplexity = exp(average loss)
    - Lower is better
    - Typical range for GPT-2 on WikiText: 20-40

    Args:
        model: The language model
        dataset: Dataset to evaluate on
        max_samples: Number of samples to evaluate (for speed)

    Returns:
        perplexity: The perplexity score
        avg_loss: The average loss
    """
    total_loss = 0
    total_tokens = 0

    # Use subset for faster evaluation
    eval_samples = min(max_samples, len(dataset))

    with torch.no_grad():  # Don't calculate gradients (faster, less memory)
        for i in tqdm(range(eval_samples), desc="Calculating perplexity"):
            sample = dataset[i]
            input_ids = torch.tensor([sample['input_ids']]).to(device)
            labels = torch.tensor([sample['labels']]).to(device)

            # Get model predictions and loss
            outputs = model(input_ids, labels=labels)
            loss = outputs.loss

            # Accumulate loss
            total_loss += loss.item() * len(sample['input_ids'])
            total_tokens += len(sample['input_ids'])

    # Calculate average loss and perplexity
    avg_loss = total_loss / total_tokens
    perplexity = np.exp(avg_loss)

    return perplexity, avg_loss

print("Perplexity calculation function defined")

In [ ]:
# CELL 9: Calculate Baseline Perplexity

print("\n" + "="*80)
print("BASELINE PERFORMANCE EVALUATION")
print("="*80)
print("\nEvaluating pre-trained GPT-2 model on WikiText-103...")
print("Evaluating on 500 validation samples...")
print("\nStarting evaluation...\n")

# Calculate baseline metrics
baseline_ppl, baseline_loss = calculate_perplexity(
    model,
    lm_datasets['validation'],
    max_samples=500
)

# Display results
print("\n" + "="*80)
print("BASELINE RESULTS")
print("="*80)
print(f"\nValidation Loss: {baseline_loss:.4f}")
print(f"Validation Perplexity: {baseline_ppl:.2f}")
print("\n--- Interpretation ---")
if baseline_ppl < 20:
    print("Excellent performance! (< 20)")
elif baseline_ppl < 40:
    print("Good performance (20-40) - typical for pre-trained GPT-2")
elif baseline_ppl < 100:
    print("Fair performance (40-100)")
else:
    print("Poor performance (> 100)")
print("="*80)

# Save baseline metrics to JSON file
baseline_metrics = {
    'model': model_name,
    'dataset': 'wikitext-103-v1',
    'block_size': block_size,
    'validation_loss': float(baseline_loss),
    'validation_perplexity': float(baseline_ppl),
    'total_parameters': total_params,
    'trainable_parameters': trainable_params,
    'evaluation_samples': 500
}

with open('baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

print("\n✓ Baseline metrics saved to baseline_metrics.json")
print("  (You can download this file from Colab's Files panel)")

In [ ]:
# CELL 10: Define Text Generation Function

def generate_text(prompt, strategy='greedy', max_length=100, **kwargs):
    """
    Generate text using different sampling strategies.

    Sampling Strategies:
    1. greedy: Always picks most likely token (deterministic)
    2. sampling: Random sampling with temperature control
    3. top_k: Only sample from top k most likely tokens
    4. top_p (nucleus): Sample from smallest set with cumulative prob > p

    Args:
        prompt: Input text to continue
        strategy: Sampling strategy ('greedy', 'sampling', 'top_k', 'top_p')
        max_length: Maximum length of generated text
        **kwargs: Strategy-specific parameters
            - temperature: Randomness (0.1=focused, 2.0=creative)
            - top_k: Number of top tokens to consider (e.g., 50)
            - top_p: Cumulative probability threshold (e.g., 0.95)

    Returns:
        Generated text as string
    """
    # Encode prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Base generation parameters
    gen_kwargs = {
        'max_length': max_length,
        'pad_token_id': tokenizer.eos_token_id
    }

    # Strategy-specific parameters
    if strategy == 'greedy':
        gen_kwargs['do_sample'] = False

    elif strategy == 'sampling':
        gen_kwargs['do_sample'] = True
        gen_kwargs['temperature'] = kwargs.get('temperature', 1.0)

    elif strategy == 'top_k':
        gen_kwargs['do_sample'] = True
        gen_kwargs['top_k'] = kwargs.get('top_k', 50)
        gen_kwargs['temperature'] = kwargs.get('temperature', 1.0)

    elif strategy == 'top_p':
        gen_kwargs['do_sample'] = True
        gen_kwargs['top_p'] = kwargs.get('top_p', 0.95)
        gen_kwargs['temperature'] = kwargs.get('temperature', 1.0)

    # Generate text
    with torch.no_grad():
        output = model.generate(input_ids, **gen_kwargs)

    # Decode and return
    return tokenizer.decode(output[0], skip_special_tokens=True)

print("Text generation function defined")

In [ ]:
# CELL 11: Generate Text with Different Sampling Strategies

print("\n" + "="*80)
print("TEXT GENERATION EXAMPLES")
print("="*80)

# Test prompt
prompt = "The history of artificial intelligence began"
print(f"\nPrompt: \"{prompt}\"\n")

# Define sampling strategies to test
strategies = [
    ('Greedy Decoding', 'greedy', {}),
    ('Temperature Sampling (T=0.8)', 'sampling', {'temperature': 0.8}),
    ('Top-k Sampling (k=50)', 'top_k', {'top_k': 50}),
    ('Nucleus Sampling (p=0.95)', 'top_p', {'top_p': 0.95})
]

# Generate text with each strategy
for name, strategy, params in strategies:
    print("="*80)
    print(f"{name}")
    print("="*80)

    generated = generate_text(
        prompt,
        strategy=strategy,
        max_length=80,
        **params
    )

    print(generated)
    print()

print("="*80)

In [ ]:
# CELL 12: Try Your Own Prompts (Interactive)

print("\n" + "="*80)
print("CUSTOM TEXT GENERATION")
print("="*80)
print("\nModify the variables below to try different prompts!\n")

# ===== MODIFY THESE VALUES =====
your_prompt = "In the field of natural language processing"
your_strategy = "top_p"  # Options: 'greedy', 'sampling', 'top_k', 'top_p'
your_params = {'top_p': 0.95, 'temperature': 0.9}
# ================================

print(f"Prompt: \"{your_prompt}\"")
print(f"Strategy: {your_strategy}")
print(f"Parameters: {your_params}\n")

generated = generate_text(
    your_prompt,
    strategy=your_strategy,
    max_length=100,
    **your_params
)

print("-" * 80)
print("GENERATED TEXT:")
print("-" * 80)
print(generated)
print("="*80)

In [ ]:
# CELL 13: Configure Training Parameters (For Week 4)

print("\n" + "="*80)
print("TRAINING CONFIGURATION (Week 4 Preview)")
print("="*80)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,  # Effective batch size: 32
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),  # Mixed precision if GPU available
    report_to="none",
)

print("\n--- Training Parameters ---")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size per device: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Warmup steps: {training_args.warmup_steps}")
print(f"Weight decay: {training_args.weight_decay}")
print(f"Mixed precision (FP16): {training_args.fp16}")

# Estimate training time
total_steps = len(lm_datasets['train']) // (
    training_args.per_device_train_batch_size *
    training_args.gradient_accumulation_steps
)
total_steps *= training_args.num_train_epochs

print(f"\n--- Training Estimates ---")
print(f"Total training steps: {total_steps:,}")
